# Handwritten Text Recognition using ANN

This notebook implements an OCR (Optical Character Recognition) pipeline for reading handwritten or printed text from images. The system:

1. **Preprocesses** images — grayscale, deskew, binarize, erode/dilate
2. **Segments** individual characters via contour detection and region merging
3. **Trains** a small fully-connected ANN (Artificial Neural Network) on the segmented characters
4. **Predicts** text for each image and evaluates using a Hamming-based distance metric

## 1. Imports

In [ ]:
import math
import os
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage
from keras.models import Sequential
from keras.layers import Dense, Input
from keras.optimizers import SGD

## 2. Configuration

In [ ]:
# Region merging thresholds — how close two contours must be to be merged into one character
MAX_DISTANCE_X = 0
MAX_DISTANCE_Y = 0

# Morphological operation parameters
DILATE_KERNEL_SIZE = 3
ERODE_KERNEL_SIZE  = 3
DILATE_ITERS       = 1
ERODE_ITERS        = 1

# Gap threshold (pixels) between characters before inserting a space
SPACE_THRESHOLD = 9

## 3. ANN Utilities

### 3.1 Data Preparation

In [ ]:
def prepare_for_ann(regions):
    """
    Normalize pixel values to [0, 1] and flatten each 28×28 region
    into a 784-element input vector.
    """
    return [region / 255.0 for region in regions]


def convert_output(alphabet):
    """
    Create one-hot encoded target vectors for each character in the alphabet.

    Returns
    -------
    np.ndarray  Shape: (len(alphabet), len(alphabet))
    """
    nn_outputs = []
    for index in range(len(alphabet)):
        output = np.zeros(len(alphabet))
        output[index] = 1
        nn_outputs.append(output)
    return np.array(nn_outputs)

### 3.2 Model Definition & Training

Architecture: `Input(784) → Dense(128, sigmoid) → Dense(output_size, sigmoid)`

Trained with SGD + momentum and MSE loss.

In [ ]:
def create_ann(output_size):
    """
    Build a simple two-layer fully-connected network.

    Parameters
    ----------
    output_size : int  Number of characters in the alphabet
    """
    ann = Sequential()
    ann.add(Input(shape=(784,)))
    ann.add(Dense(128, activation='sigmoid'))
    ann.add(Dense(output_size, activation='sigmoid'))
    return ann


def train_ann(ann, X_train, y_train, epochs):
    """
    Train the ANN using SGD with momentum.

    Parameters
    ----------
    ann      : keras.Sequential  Model to train
    X_train  : list              Flattened & normalised character images
    y_train  : list              One-hot target vectors
    epochs   : int               Number of training epochs

    Returns
    -------
    keras.Sequential  Trained model
    """
    X_train = np.array(X_train, np.float32)
    y_train = np.array(y_train, np.float32)
    sgd = SGD(learning_rate=0.01, momentum=0.9)
    ann.compile(loss='mean_squared_error', optimizer=sgd)
    ann.fit(X_train, y_train, epochs=epochs, batch_size=1, verbose=0, shuffle=False)
    return ann


def winner(output):
    """Return the index of the highest-confidence output neuron."""
    return max(enumerate(output), key=lambda x: x[1])[0]

### 3.3 Evaluation Metric

Uses a **Hamming-based distance** — counts mismatched characters at each position, plus the difference in string length for insertions/deletions.

In [ ]:
def calculate_distance(target, predicted):
    """
    Compute a character-level error distance between two strings.
    Combines positional mismatches and length difference.

    Parameters
    ----------
    target    : any  Ground truth value (converted to str)
    predicted : any  Predicted value (converted to str)

    Returns
    -------
    int  Total error distance
    """
    target, predicted = str(target), str(predicted)
    min_len   = min(len(target), len(predicted))
    ham_dist  = sum(1 for i in range(min_len) if target[i] != predicted[i])
    return ham_dist + abs(len(target) - len(predicted))

## 4. Image Processing Pipeline

### 4.1 Deskewing

Uses the Probabilistic Hough Line Transform to estimate the dominant text angle, then rotates the image to correct for skew.

In [ ]:
def deskew_img(image, angles=None):
    """
    Detect and correct image skew using Hough Line Transform.

    Parameters
    ----------
    image  : np.ndarray       Grayscale or binary image
    angles : list | None      Pre-computed angles (reused for color image pass)

    Returns
    -------
    rotated : np.ndarray  Deskewed image
    angles  : list        Detected line angles (in degrees)
    """
    if angles is None:
        img_edges = cv2.Canny(image, threshold1=40, threshold2=80)
        lines     = cv2.HoughLinesP(img_edges, rho=1, theta=math.pi / 180.0,
                                    threshold=40, maxLineGap=10)
        angles = []
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                angles.append(math.degrees(math.atan2(y2 - y1, x2 - x1)))

    median_angle = np.median(angles) if len(angles) > 0 else 0
    rotated      = ndimage.rotate(image, median_angle, cval=0, reshape=True)
    return np.uint8(rotated), angles

### 4.2 Region Merging

Some characters produce multiple disconnected contours (e.g. dotted letters). This function merges contours that are spatially close into a single bounding box.

In [ ]:
def merge_nearby_regions(regions_array, max_distance_x=10, max_distance_y=15):
    """
    Merge contour bounding boxes that are close enough to belong to the same character.

    Parameters
    ----------
    regions_array  : list  Each element is [region_img, (x, y, w, h)]
    max_distance_x : int   Max horizontal gap to still merge
    max_distance_y : int   Max vertical gap to still merge

    Returns
    -------
    list  Merged regions in the same format
    """
    if len(regions_array) == 0:
        return []

    regions_array = sorted(regions_array, key=lambda x: x[1][0])
    merged        = []
    skip_indices  = set()

    for i in range(len(regions_array)):
        if i in skip_indices:
            continue

        current_rect     = regions_array[i][1]
        regions_to_merge = [i]

        for j in range(i + 1, len(regions_array)):
            if j in skip_indices:
                continue

            next_rect = regions_array[j][1]
            x_dist    = next_rect[0] - (current_rect[0] + current_rect[2])

            y1_curr, y2_curr = current_rect[1], current_rect[1] + current_rect[3]
            y1_next, y2_next = next_rect[1],    next_rect[1]    + next_rect[3]

            if   y2_curr < y1_next: y_dist = y1_next - y2_curr
            elif y2_next < y1_curr: y_dist = y1_curr - y2_next
            else:                   y_dist = 0

            if x_dist <= max_distance_x and y_dist <= max_distance_y:
                regions_to_merge.append(j)
                skip_indices.add(j)

        if len(regions_to_merge) == 1:
            merged.append(regions_array[i])
        else:
            rects   = [regions_array[idx][1] for idx in regions_to_merge]
            min_x   = min(r[0] for r in rects)
            min_y   = min(r[1] for r in rects)
            max_x   = max(r[0] + r[2] for r in rects)
            max_y   = max(r[1] + r[3] for r in rects)
            merged.append([None, (min_x, min_y, max_x - min_x, max_y - min_y)])

    return merged

### 4.3 ROI Extraction with Inter-Character Distances

In [ ]:
def select_roi_with_distances(image_bin):
    """
    Detect character regions from a binary image, merge nearby contours,
    resize each to 28×28, and compute inter-character gaps.

    Parameters
    ----------
    image_bin : np.ndarray  Binary image (white text on black background)

    Returns
    -------
    sorted_regions    : list  28×28 binary region images, sorted left to right
    region_distances  : list  Pixel gaps between consecutive regions
    """
    contours, _ = cv2.findContours(
        image_bin.copy(), mode=cv2.RETR_EXTERNAL, method=cv2.CHAIN_APPROX_SIMPLE
    )
    regions_array = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        region = image_bin[y:y+h+1, x:x+w+1]
        regions_array.append([region, (x, y, w, h)])

    regions_array = merge_nearby_regions(
        regions_array, max_distance_x=MAX_DISTANCE_X, max_distance_y=MAX_DISTANCE_Y
    )

    # Fill in merged region images from the binary image
    for i, (region, rect) in enumerate(regions_array):
        if region is None:
            x, y, w, h = rect
            regions_array[i][0] = image_bin[y:y+h+1, x:x+w+1]

    # Resize all regions to 28×28
    for i in range(len(regions_array)):
        regions_array[i][0] = cv2.resize(
            regions_array[i][0], dsize=(28, 28), interpolation=cv2.INTER_NEAREST
        )

    regions_array   = sorted(regions_array, key=lambda x: x[1][0])
    sorted_regions  = [r[0] for r in regions_array]
    sorted_rects    = [r[1] for r in regions_array]

    region_distances = [
        sorted_rects[i+1][0] - (sorted_rects[i][0] + sorted_rects[i][2])
        for i in range(len(sorted_rects) - 1)
    ]

    return sorted_regions, region_distances

### 4.4 Full Image Preprocessing

In [ ]:
def process_image(img):
    """
    Full preprocessing pipeline for a single image:
    grayscale → deskew → binarize → erode → dilate → segment.

    Parameters
    ----------
    img : np.ndarray  RGB input image

    Returns
    -------
    regions   : list  Segmented 28×28 character images
    distances : list  Pixel gaps between consecutive characters
    """
    img_gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    img_rotated_gs, angles = deskew_img(img_gray)
    img_rotated_gs         = np.uint8(img_rotated_gs)

    _, img_bin = cv2.threshold(img_rotated_gs, 127, 255, cv2.THRESH_BINARY)

    erode_kernel = np.ones((ERODE_KERNEL_SIZE,  ERODE_KERNEL_SIZE),  np.uint8)
    dilate_kernel= np.ones((DILATE_KERNEL_SIZE, DILATE_KERNEL_SIZE), np.uint8)

    img_bin = cv2.erode(img_bin,  erode_kernel,  iterations=ERODE_ITERS)
    img_bin = cv2.dilate(img_bin, dilate_kernel, iterations=DILATE_ITERS)

    regions, distances = select_roi_with_distances(img_bin)
    return regions, distances

### 4.5 Text Assembly with Space Insertion

In [ ]:
def display_result_with_spaces(outputs, alphabet, distances, threshold=10):
    """
    Assemble predicted characters into a string, inserting a space
    wherever the inter-character gap exceeds the threshold.

    Parameters
    ----------
    outputs   : list  ANN output vectors, one per character region
    alphabet  : list  Sorted list of unique characters
    distances : list  Pixel gaps between consecutive characters
    threshold : int   Gap size above which a space is inserted

    Returns
    -------
    str  Assembled predicted text
    """
    if len(outputs) == 0:
        return ""
    if len(distances) == 0:
        return alphabet[winner(outputs[0])]

    result = alphabet[winner(outputs[0])]
    for i in range(1, len(outputs)):
        if distances[i-1] > threshold:
            result += " "
        result += alphabet[winner(outputs[i])]
    return result

## 5. Training

### 5.1 Load Dataset

In [ ]:
DATA_DIR = "data"

df = pd.read_csv(os.path.join(DATA_DIR, 'texts.csv'), sep='|')
print(f"Loaded {len(df)} samples.")
print(df.head())

### 5.2 Build Alphabet & One-Hot Encoding

In [ ]:
full_text      = "".join(df['text'].values)
alphabet       = sorted(list(set(full_text.replace(" ", ""))))
char_to_onehot = convert_output(alphabet)

print(f"Alphabet size: {len(alphabet)}")
print(f"Characters:    {''.join(alphabet)}")

### 5.3 Preprocess All Images

In [ ]:
imgs, texts, processed_images = [], [], []

for _, row in df.iterrows():
    img = cv2.cvtColor(cv2.imread(os.path.join(DATA_DIR, row['image'])), cv2.COLOR_BGR2RGB)
    imgs.append(img)
    texts.append(row['text'])

for img, true_text in zip(imgs, texts):
    regions, distances = process_image(img)
    processed_images.append((regions, distances))

print("Preprocessing complete.")

### 5.4 Prepare Training Data

Only images where the number of detected regions matches the number of characters in the ground truth are used for training — mismatches indicate segmentation errors.

In [ ]:
X_train_all, y_train_all = [], []
skipped = 0

for true_text, (regions, _) in zip(texts, processed_images):
    true_text_clean = true_text.replace(" ", "")
    if len(regions) == len(true_text_clean):
        prepared = prepare_for_ann(regions)
        for i, p in enumerate(prepared):
            X_train_all.append(p.flatten())
            y_train_all.append(char_to_onehot[alphabet.index(true_text_clean[i])])
    else:
        skipped += 1

print(f"Training samples : {len(X_train_all)}")
print(f"Skipped (mismatch): {skipped}")

### 5.5 Train the ANN

In [ ]:
ann = None

if len(X_train_all) > 0:
    ann = create_ann(len(alphabet))
    ann.summary()
    ann = train_ann(ann, X_train_all, y_train_all, epochs=800)
    print("Training complete.")
else:
    print("WARNING: No training samples available.")

## 6. Evaluation

In [ ]:
total_dist = 0
results    = []

for img, true_text, (regions, distances) in zip(imgs, texts, processed_images):
    if regions and ann is not None:
        prepared  = prepare_for_ann(regions)
        outputs   = ann.predict(np.array([p.flatten() for p in prepared], np.float32), verbose=0)
        pred_text = display_result_with_spaces(outputs, alphabet, distances, threshold=SPACE_THRESHOLD)
    else:
        pred_text = ""

    d = calculate_distance(true_text, pred_text)
    total_dist += d
    results.append({'true': true_text, 'predicted': pred_text, 'distance': d})

print(f"Total distance : {total_dist}")
print(f"Mean distance  : {total_dist / len(results):.2f}")

### 6.1 Results Table

In [ ]:
print(f"{'True Text':<30} {'Predicted':<30} {'Distance':>8}")
print("-" * 72)
for r in results:
    print(f"{r['true']:<30} {r['predicted']:<30} {r['distance']:>8}")
print("-" * 72)
print(f"{'Total':<60} {total_dist:>8}")

## 7. Visualisation

### 7.1 Segmented Characters for One Image

In [ ]:
SAMPLE_IDX = 0  # Change to inspect a different image

sample_img   = imgs[SAMPLE_IDX]
sample_text  = texts[SAMPLE_IDX]
regions, _   = processed_images[SAMPLE_IDX]

print(f"Ground truth : '{sample_text}'")
print(f"Regions found: {len(regions)}  |  Characters: {len(sample_text.replace(' ', ''))}")

n = len(regions)
if n > 0:
    cols = min(n, 20)
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.5))
    axes = np.array(axes).flatten()
    for i, region in enumerate(regions):
        axes[i].imshow(region, cmap='gray')
        axes[i].axis('off')
    for i in range(n, len(axes)):
        axes[i].axis('off')
    plt.suptitle(f"Segmented characters — '{sample_text}'")
    plt.tight_layout()
    plt.show()

### 7.2 Error Distance per Sample

In [ ]:
distances_plot = [r['distance'] for r in results]
labels         = [r['true'][:15] for r in results]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(range(len(distances_plot)), distances_plot,
              color=['green' if d == 0 else 'coral' for d in distances_plot])
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Edit Distance')
ax.set_title(f'Per-Sample Error  (Total = {total_dist})')
plt.tight_layout()
plt.show()